# **Cell Convert & Split**

In [ ]:
import os
import json
import glob
import random
from shapely import wkt
from tqdm import tqdm

random.seed(42)

# --- Konfigurasi ---
TRAIN_BASE  = '/kaggle/input/datasets/trsnwn/xbd-train1/train'
TEST_BASE   = '/kaggle/input/datasets/trsnwn/xbd-test1/test'
OUTPUT_BASE = '/kaggle/working/xbd_yolo'
IMG_SIZE    = 1024

DAMAGE_MAP = {'minor-damage': 0, 'major-damage': 1, 'destroyed': 2}

for split in ['train', 'val', 'test']:
    os.makedirs(f'{OUTPUT_BASE}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUTPUT_BASE}/{split}/labels', exist_ok=True)


def polygon_wkt_to_bbox_yolo(wkt_str):
    poly = wkt.loads(wkt_str)
    minx, miny, maxx, maxy = poly.bounds
    cx = ((minx + maxx) / 2) / IMG_SIZE
    cy = ((miny + maxy) / 2) / IMG_SIZE
    bw = (maxx - minx) / IMG_SIZE
    bh = (maxy - miny) / IMG_SIZE
    return cx, cy, bw, bh


def convert_split(base_path, split_name, val_ratio=0.2):
    label_dir = os.path.join(base_path, 'labels')
    image_dir = os.path.join(base_path, 'images')

    json_files = sorted(glob.glob(os.path.join(label_dir, '*_post_disaster.json')))
    print(f"\n[{split_name}] Ditemukan {len(json_files)} file JSON")

    if split_name == 'train':
        random.shuffle(json_files)
        cut = int(len(json_files) * (1 - val_ratio))
        subsets = [('train', json_files[:cut]), ('val', json_files[cut:])]
    else:
        subsets = [('test', json_files)]

    # hitung jumlah objek tiap kelas per subset
    counts = {out_split: {k: 0 for k in DAMAGE_MAP} for out_split, _ in subsets}

    for out_split, files in subsets:
        for json_path in tqdm(files, desc=f'  -> {out_split}'):
            with open(json_path) as f:
                data = json.load(f)

            basename     = os.path.basename(json_path).replace('_post_disaster.json', '')
            img_filename = basename + '_post_disaster.png'
            img_src      = os.path.join(image_dir, img_filename)

            yolo_lines = []
            for feat in data['features']['xy']:
                subtype = feat['properties']['subtype'].strip().lower()
                if subtype not in DAMAGE_MAP:
                    continue
                cx, cy, bw, bh = polygon_wkt_to_bbox_yolo(feat['wkt'])
                yolo_lines.append(f"{DAMAGE_MAP[subtype]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                counts[out_split][subtype] += 1

            txt_path = f"{OUTPUT_BASE}/{out_split}/labels/{basename}_post_disaster.txt"
            with open(txt_path, 'w') as f:
                f.write('\n'.join(yolo_lines))

            img_dst = f"{OUTPUT_BASE}/{out_split}/images/{img_filename}"
            if not os.path.exists(img_dst):
                os.symlink(img_src, img_dst)

    # cetak distribusi kelas tiap subset
    for out_split in counts:
        print(f"\n  Distribusi kelas ({out_split}):")
        for cls, n in counts[out_split].items():
            print(f"    {cls:<15}: {n:,}")


convert_split(TRAIN_BASE, 'train')
convert_split(TEST_BASE, 'test')
print("\nKonversi selesai.")

In [ ]:
!pip install ultralytics

In [ ]:
!pip install -q timm

# **Cell Adapter Timm**

In [ ]:
import torch.nn as nn
import timm
from ultralytics.nn.modules import TorchVision
 
MODEL_NAME = "tiny_vit_21m_224"
OUT_INDICES = (1, 2, 3) 
 
_original_init = TorchVision.__init__
_original_forward = TorchVision.forward
 
def _patched_init(self, model, weights="DEFAULT", unwrap=True, truncate=2, split=False):
    if model == MODEL_NAME:
        nn.Module.__init__(self)
        self.m = timm.create_model(model, pretrained=True, features_only=True, out_indices=OUT_INDICES)
        self._is_timm = True
        print(f"[timm_backbone] {model} -> feature_channels={self.m.feature_info.channels()}")
    else:
        self._is_timm = False
        _original_init(self, model, weights, unwrap, truncate, split)
 
def _patched_forward(self, x):
    if getattr(self, "_is_timm", False):
        return self.m(x)
    return _original_forward(self, x)
 
TorchVision.__init__ = _patched_init
TorchVision.forward = _patched_forward
 
print(f"Patch timm backbone aktif untuk: {MODEL_NAME}")

# **Cell Load Model dari TorchVision**

In [ ]:
from ultralytics import YOLO
import torch

yaml_content = """
nc: 3
backbone:
  - [-1, 1, TorchVision, [576, tiny_vit_21m_224, IMAGENET1K_V1, True, 2, True]]
  - [0, 1, Index, [192, 0]]   # P3 (stage asli index 1, reduction=8)
  - [0, 1, Index, [384, 1]]   # P4 (stage asli index 2, reduction=16)
  - [0, 1, Index, [576, 2]]   # P5 (stage asli index 3, reduction=32)
  - [1, 1, Conv, [64, 1, 1]]
  - [2, 1, Conv, [128, 1, 1]]
  - [3, 1, Conv, [256, 1, 1]]

head:
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 1, C2f, [128]]
  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 1, C2f, [64]]
  - [-1, 1, Conv, [64, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 1, C2f, [128]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 1, C2f, [256]]
  - [[12, 15, 18], 1, Detect, [nc]]
"""

yaml_path = "/kaggle/working/yolov8n_tinyvit.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"YAML tersimpan di: {yaml_path}")

# **Cell Train**

In [ ]:
from ultralytics import YOLO

torch.backends.cuda.enable_mem_efficient_sdp(False)

DATA_YAML   = "/kaggle/working/xbd_yolo/data.yaml"
PROJECT_DIR = "/kaggle/working/runs"
RUN_NAME    = "yolov8n_tinyvit"

with open(DATA_YAML, "w") as f:
    f.write("""path: /kaggle/working/xbd_yolo
train: train/images
val: val/images
test: test/images

nc: 3
names: ['minor-damage', 'major-damage', 'destroyed']
""")

model = YOLO(yaml_path)
try:
    results = model.train(
        data=DATA_YAML,
        project=PROJECT_DIR,
        name=RUN_NAME,

        epochs=100,
        patience=0,
        imgsz=640,
        batch=8,
        optimizer="SGD",
        lr0=0.01,
        lrf=0.0001,

        fliplr=0.5,
        flipud=0.5,
        scale=0.5,
        mosaic=1.0,
        mixup=0.1,

        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.0,
        translate=0.0,

        save=True,
        save_period=-1,
        plots=True,
        verbose=True,
        exist_ok=True,
    )

    print(f"\nTraining selesai. Hasil tersimpan di: {PROJECT_DIR}/{RUN_NAME}")

except RuntimeError as e:
    print(f"\n[PERINGATAN] Training selesai, tapi ada error saat proses berjalan: {e}")

# **Cell Test Set**

In [ ]:
import os
from ultralytics import YOLO

RUN_DIR = '/kaggle/working/runs/yolov8n_tinyvit'
PROJECT_DIR, RUN_NAME = os.path.split(RUN_DIR)
CLASS_NAMES = ['minor-damage', 'major-damage', 'destroyed']
IOU_THRESHOLD = 0.5

model = YOLO(f'{RUN_DIR}/weights/best.pt')
metrics = model.val(
    data='/kaggle/working/xbd_yolo/data.yaml',
    split='test',
    imgsz=640,
    iou=IOU_THRESHOLD,
    device='cpu',
    plots=True,
    project=PROJECT_DIR,   # arahkan plot ke folder training yang sama
    name=RUN_NAME,         # (RUN_DIR), bukan folder default runs/detect/val
    exist_ok=True,         # timpa isi folder yang sudah ada, jangan buat
                            # yolov8n_tinyvit2 atau semacamnya
)

cm = metrics.confusion_matrix.matrix
iou_per_class = []
for i in range(len(CLASS_NAMES)):
    tp = cm[i, i]
    fp = cm[i, :].sum() - tp
    fn = cm[:, i].sum() - tp
    denom = tp + fp + fn
    iou = (tp / denom) if denom > 0 else 0.0
    iou_per_class.append(iou)

print(f"\n=== Hasil Evaluasi Test Set: yolov8n_tinyvit ===")
for i, kelas in enumerate(CLASS_NAMES):
    print(f"{kelas:<15} | Precision={metrics.box.p[i]:.4f}  Recall={metrics.box.r[i]:.4f}  "
          f"mAP50={metrics.box.ap50[i]:.4f}  mAP50-95={metrics.box.maps[i]:.4f}  "
          f"F1={metrics.box.f1[i]:.4f}  IoU={iou_per_class[i]:.4f}")

print(f"{'Rata-rata':<15} | Precision={metrics.box.p.mean():.4f}  Recall={metrics.box.r.mean():.4f}  "
      f"mAP50={metrics.box.ap50.mean():.4f}  mAP50-95={metrics.box.maps[:len(CLASS_NAMES)].mean():.4f}  "
      f"F1={metrics.box.f1.mean():.4f}  IoU={sum(iou_per_class) / len(iou_per_class):.4f}")

print(f"\nIsi folder {RUN_DIR} sekarang:")
print(os.listdir(RUN_DIR))

In [ ]:
from IPython.display import Image, display
import os

RUN_DIR = '/kaggle/working/runs/yolov8n_tinyvit'

for fname in ['BoxPR_curve.png', 'BoxF1_curve.png', 'BoxP_curve.png',
              'BoxR_curve.png', 'confusion_matrix_normalized.png']:
    fpath = os.path.join(RUN_DIR, fname)
    if os.path.exists(fpath):
        print(fname)
        display(Image(fpath))
    else:
        print(f"[!] Tidak ditemukan: {fpath}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

RUN_DIR = '/kaggle/working/runs/yolov8n_tinyvit'
csv_path = os.path.join(RUN_DIR, 'results.csv')

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]

kolom_dipakai = [
    'train/box_loss', 'train/cls_loss',
    'metrics/precision(B)', 'metrics/recall(B)',
    'val/box_loss', 'val/cls_loss',
]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()

for i, kolom in enumerate(kolom_dipakai):
    axes[i].plot(df['epoch'], df[kolom], marker='o')
    axes[i].set_title(kolom)
    axes[i].set_xlabel('epoch')

plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'results_manual.png'), dpi=150)
plt.show()